- Link dataset: https://www.kaggle.com/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import seaborn as sns
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import matplotlib.pyplot as plt
from sklearn import neighbors
from sklearn.ensemble import RandomForestClassifier
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Restaurants_Train_v2.csv", encoding='utf8')
# df_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Restaurants_Test_Data_PhaseA.csv", encoding='utf8')
df = pd.read_csv("/kaggle/input/semeval-2014-task-4-aspectbasedsentimentanalysis/Restaurants_Train_v2.csv", encoding='utf8')

In [ ]:
labels = df['polarity'].unique().tolist()
label_map = {label: i for i, label in enumerate(labels)}
num_labels = len(labels)

df['labels'] = df['polarity'].map(label_map)

print(f"Number of labels: {num_labels}")
print(f"Label mapping: {label_map}")

In [ ]:
label_names = ['negative', 'positive', 'neutral']

In [ ]:
# df_merge = df['Sentence'] + ' [asp] ' + df['Aspect Term']
# df = pd.concat([df, df_merge.rename('sentence')], axis=1)

In [ ]:
df = df.drop(['polarity', 'id', 'from', 'to'], axis=1)

In [ ]:
df = df[df['labels'] != 3]
df

# **Augment Data**

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
# stratify=df['labels'])

In [ ]:
import matplotlib.pyplot as plt

def plot_sentiment_distribution(train_df, val_df):

    # Count sentiments in train, test sets
    train_counts = train_df.value_counts().sort_index()
    test_counts = val_df.value_counts().sort_index()

    # Set up the plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot bars
    x = range(len(train_counts))
    width = 0.35
    ax.bar([i - width/2 for i in x], train_counts.values, width, label='Train', alpha=0.8, color='#3498db')
    ax.bar([i + width/2 for i in x], test_counts.values, width, label='Test', alpha=0.8, color='#e74c3c')

    # Customize the plot
    ax.set_ylabel('Số lượng mẫu')
    ax.set_title('Phân bổ cảm xúc trong tập Train và Test')
    ax.set_xticks(x)

    ax.set_xticklabels(label_names)

    ax.legend()

    # Add value labels
    for i, v in enumerate(train_counts.values):
        ax.text(i - width/2, v, str(v), ha='center', va='bottom')
    for i, v in enumerate(test_counts.values):
        ax.text(i + width/2, v, str(v), ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

plot_sentiment_distribution(y_train, y_test)

In [ ]:
# import re
# from nltk.corpus import stopwords, wordnet
# from nltk.stem import WordNetLemmatizer
# import random
# import nltk
# import warnings
# warnings.filterwarnings('ignore')

In [ ]:
# stop_words = set(stopwords.words("english"))

# def get_synonyms(word):
#     synonyms = set()
#     for syn in wordnet.synsets(word):
#         for lemma in syn.lemmas():
#             synonym = lemma.name().replace("_", " ").lower()
#             if synonym != word.lower():
#                 synonyms.add(synonym)
#     return list(synonyms)


# def synonym_replacement(sentence, n=2):
#     words = sentence.split()
#     new_words = words.copy()

#     # chỉ chọn từ không phải stopword
#     candidates = [w for w in words if w.lower() not in stop_words]

#     if len(candidates) == 0:
#         return sentence

#     random_words = random.sample(candidates, min(n, len(candidates)))

#     for word in random_words:
#         synonyms = get_synonyms(word)
#         if synonyms:
#             synonym = random.choice(synonyms)
#             idx = words.index(word)
#             new_words[idx] = synonym

#     return " ".join(new_words)

In [ ]:
# Áp dụng tăng cường dữ liệu cho cột 'Sentence'
# df['Sentence_augmented'] =df['Sentence']

# for i in df.index:
#     ss=synonym_replacement(df['Sentence'][i])
#     df['Sentence_augmented'][i] = ss.split()

    
    
    #df['Sentence'].apply(lambda x: synonym_replacement(x))

# **Processing Data**

## Lower

In [ ]:
df

In [ ]:
train_df['Sentence'] = train_df['Sentence'].str.lower()
val_df['Sentence'] = val_df['Sentence'].str.lower()

## Loại bỏ punc

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def remove_punc_spacy(text):
    doc = nlp(text)
    return " ".join([token.text for token in doc if not token.is_punct])

## Lemmatizaion

In [ ]:
def func_lemma(text):
    doc = nlp(text)
    
    tmp = ""
    
    for token in doc:
        tmp += token.lemma_ + " "
    
    return tmp.strip()

## Stopwords

In [ ]:
KEEP_WORDS = [
    # negation
    'not', 'no', 'never', 'none', 'nobody', 'nothing',
    'neither', 'nor', 'cannot', 'without',
    "n't", 'n’t', 'ca', 'noone',

    # intensity
    'very', 'too', 'so', 'quite', 'really',
    'most', 'least', 'less', 'much', 'enough',
    'almost', 'pretty',

    # contrast
    'but', 'however', 'although', 'though',
    'nevertheless', 'yet', 'whereas',

    # frequency
    'always', 'often', 'sometimes', 'ever'
]


In [ ]:

nlp = spacy.load("en_core_web_sm")

stop_words = nlp.Defaults.stop_words

In [ ]:
sw = [word for word in stop_words if word not in KEEP_WORDS]
print("stopword ban đầu: ", len(stop_words))
print("stopword sau khi loại bỏ: ", len(sw))

In [ ]:
def remove_stopwords(text):
    word_list = text.split()
    filtered_words = [word for word in word_list if word not in sw]
    filtered_text = ' '.join(filtered_words)
    return filtered_text

remove_stopwords("I am running a marathon and I have run many before.")

In [ ]:
train_df['Sentence'] = train_df['Sentence'].apply(remove_stopwords)
train_df['Sentence'] = train_df['Sentence'].apply(func_lemma)
train_df['Sentence'] = train_df['Sentence'].apply(remove_punc_spacy)

In [ ]:
val_df['Sentence'] = val_df['Sentence'].apply(remove_stopwords)
val_df['Sentence'] = val_df['Sentence'].apply(func_lemma)
val_df['Sentence'] = val_df['Sentence'].apply(remove_punc_spacy)

# 2. **Vector hóa**

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # unigram + bigram
    # max_features=15000,    # giới hạn vocab
)
X_train_vectors = vectorizer.fit_transform(train_df['Sentence'])
X_test_vectors = vectorizer.transform(val_df['Sentence'])

In [ ]:
# Confusion matrix
def p_cm(y_pred_name_model):
    cm = confusion_matrix(y_test, y_pred_name_model)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names,
                yticklabels=label_names)

    plt.title('Confusion Matrix')
    plt.ylabel('Thực tế (Actual)')
    plt.xlabel('Dự đoán (Predicted)')
    plt.show()

# Naive Bayes

In [ ]:
model = MultinomialNB()

model.fit(X_train_vectors, y_train)

In [ ]:
y_pred_nb = model.predict(X_test_vectors)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))

In [ ]:
p_cm(y_pred_nb)

# Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42) # class_weight='balanced' => Không tốt, khả năng dữ liệu test cũng bị lệch nhiều

model.fit(X_train_vectors, y_train)

In [ ]:
y_pred_lr = model.predict(X_test_vectors)

accuracy = accuracy_score(y_test, y_pred_lr)
print(f"Độ chính xác của mô hình: {accuracy}")

lr_report = classification_report(y_test, y_pred_lr, output_dict = True)
print(classification_report(y_test, y_pred_lr))

In [ ]:
p_cm(y_pred_lr)

# SVM

In [ ]:

# Scikit-learn tự động hiểu y_train có 4 lớp và áp dụng OvO
model = SVC(kernel='linear', decision_function_shape='ovo')
model.fit(X_train_vectors, y_train)

In [ ]:
y_pred_svm = model.predict(X_test_vectors)

accuracy = accuracy_score(y_test, y_pred_svm)
print(f"Độ chính xác của mô hình: {accuracy}")

svm_report = classification_report(y_test, y_pred_svm, output_dict = True)
print(classification_report(y_test, y_pred_svm))

In [ ]:
p_cm(y_pred_svm)

# KNN

In [ ]:
def myweight(distances):
    sigma2 = 0.3
    return np.exp(-distances ** 2 / sigma2)

In [ ]:
model = neighbors.KNeighborsClassifier(n_neighbors = 7, p = 2, weights = myweight)
model.fit(X_train_vectors, y_train)

In [ ]:
y_pred_knn = model.predict(X_test_vectors)

accuracy = accuracy_score(y_test, y_pred_knn)
print(f"Độ chính xác của mô hình: {accuracy}")

knn_report = classification_report(y_test, y_pred_knn, output_dict = True)
print(classification_report(y_test, y_pred_knn))

In [ ]:
p_cm(y_pred_knn)

# Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vectors, y_train)

In [ ]:
y_pred_rd = model.predict(X_test_vectors)

accuracy = accuracy_score(y_test, y_pred_rd)
print(f"Độ chính xác của mô hình: {accuracy}")

rd_report = classification_report(y_test, y_pred_rd, output_dict = True)
print(classification_report(y_test, y_pred_rd))

In [ ]:
p_cm(y_pred_rd)

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression', 'SVM', 'KNN', 'Random Forest'],
    'Accuracy': [nb_report['accuracy'], lr_report['accuracy'], svm_report['accuracy'], knn_report['accuracy'], rd_report['accuracy']],
    'Precision': [nb_report['weighted avg']['precision'], lr_report['weighted avg']['precision'], svm_report['weighted avg']['precision'], knn_report['weighted avg']['precision'], rd_report['weighted avg']['precision']],
    'Recall': [nb_report['weighted avg']['recall'], lr_report['weighted avg']['recall'], svm_report['weighted avg']['recall'], knn_report['weighted avg']['recall'], rd_report['weighted avg']['recall']],
    'F1-score': [nb_report['weighted avg']['f1-score'], lr_report['weighted avg']['f1-score'], svm_report['weighted avg']['f1-score'], knn_report['weighted avg']['f1-score'], rd_report['weighted avg']['f1-score']]
})

results_melted = pd.melt(results, id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x='Score', y='Metric', hue='Model')
plt.title('Bar Plot Comparison of Naive Bayes and Logistic Regression Models')
plt.xlabel('Score')
plt.ylabel('Metric')
plt.xlim(0, 1) 
plt.legend(title='Model')
plt.show()

In [ ]:
df.head()

In [ ]:
lengths = df['Sentence'].str.split().str.len()

print("Min length:", lengths.min())
print("Max length:", lengths.max())
print("Avg length:", lengths.mean())


# BERT

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class AspectSentimentDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):

        review = str(self.data.loc[idx, 'Sentence'])
        aspect = str(self.data.loc[idx, 'Aspect Term'])
        label = self.data.loc[idx, 'labels']

        encoding = self.tokenizer(
            review,
            aspect,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = AspectSentimentDataset(train_df, tokenizer, MAX_LEN)
val_dataset = AspectSentimentDataset(val_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.to(device)

# ========================
# Optimizer
# ========================

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

In [ ]:
# ========================
# Training loop
# ========================
best_f1 = 0
best_epoch = 0
best_preds = None
best_labels = None

for epoch in range(5):

    model.train()
    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} loss:", total_loss)

    # ========================
    # Validation
    # ========================

    model.eval()

    preds = []
    true_labels = []
    
    val_loss = 0
    
    with torch.no_grad():
    
        for batch in val_loader:
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
    
            loss = outputs.loss
            val_loss += loss.item()
    
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)
    
            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    # Metrics
    accuracy = accuracy_score(true_labels, preds)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        preds,
        average='macro'
    )
    
    avg_val_loss = val_loss / len(val_loader)
    
    print("\nValidation Results")
    print("------------------")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

    # Save best model
    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch + 1
        best_preds = preds
        best_labels = true_labels
    
        torch.save(model.state_dict(), "best_bert_model.pt")
    
        print("🔥 New best model saved!")

from sklearn.metrics import classification_report

print("\n==============================")
print("BEST MODEL RESULTS")
print("==============================")

print(f"Best Epoch: {best_epoch}")
print(f"Best F1-score: {best_f1:.4f}")

print("\nClassification Report:")
print(classification_report(best_labels, best_preds))

In [ ]:
bert_report = classification_report(best_labels, best_preds, output_dict=True)
results.loc[len(results)] = [
    'BERT',
    bert_report['accuracy'],
    bert_report['weighted avg']['precision'],
    bert_report['weighted avg']['recall'],
    bert_report['weighted avg']['f1-score']
]

In [ ]:
results_melted = pd.melt(results, id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x='Score', y='Metric', hue='Model')
plt.title('Bar Plot Comparison of Naive Bayes and Logistic Regression Models')
plt.xlabel('Score')
plt.ylabel('Metric')
plt.xlim(0, 1) 
plt.legend(title='Model')
plt.show()